# ThermoCampus — стартовое решение

Автор: укажите свои ФИО. В `train.csv` есть ответы для обучения, а `test.csv` содержит только признаки для финальных прогнозов.

## Словарь ThermoCampus 2026 v1

Строка — здание в один из 16 наблюдавшихся дней.

| Поле | Значение |
|---|---|
| row_id | Случайный непрозрачный ID строки, ключ прогнозов |
| building_id | ID здания для группового разбиения |
| building_type | residential — жилое, office — офисное, education — учебное |
| wall_type | brick — кирпич, panel — панель, monolith — монолит |
| control_type | manual — ручное, weather — погодное, smart — адаптивное управление |
| floor_area_m2 | Общая площадь, м² |
| age_years | Возраст здания, лет |
| floors | Этажность |
| insulation_mm | Толщина утепления, мм; возможны пропуски |
| glazing_ratio | Доля остекления наружной поверхности, 0–1 |
| boiler_efficiency | КПД отопительной установки, 0–1 |
| ventilation_ach | Воздухообмен, 1/час; возможны пропуски |
| day_of_season | Номер дня условного отопительного сезона, 1–150 |
| outdoor_temp_c | Средняя наружная температура, °C |
| wind_m_s | Средняя скорость ветра, м/с |
| solar_kwh_m2 | Суточная солнечная радиация, кВт·ч/м²; возможны пропуски |
| humidity_pct | Относительная влажность, %; возможны пропуски |
| is_weekend | 1 — выходной день, 0 — будний |
| occupancy_ratio | Доля присутствующих от типовой вместимости, 0–1 |
| setpoint_c | Заданная температура внутри, °C |
| heat_kwh_m2 | Цель: суточное удельное теплопотребление, кВт·ч/м²; только в train |



In [ ]:
import json, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Ridge
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_squared_error
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
print('Train / test:', train.shape, test.shape)
print('Train buildings:', train.building_id.nunique())
print('Target:', train.heat_kwh_m2.describe().round(3).to_dict())
print('Missing:', train.isna().sum()[lambda s: s > 0].to_dict())

## Валидация


In [ ]:
X = train.drop(columns=['row_id', 'building_id', 'heat_kwh_m2'])
y = train.heat_kwh_m2
num = X.select_dtypes(include='number').columns.tolist()
cat = X.select_dtypes(include='object').columns.tolist()
def make_model(estimator):
    preprocess = ColumnTransformer([
        ('num', make_pipeline(SimpleImputer(strategy='median', add_indicator=True), StandardScaler()), num),
        ('cat', make_pipeline(SimpleImputer(strategy='most_frequent'), OneHotEncoder(handle_unknown='ignore', sparse_output=False)), cat)])
    return make_pipeline(preprocess, estimator)
tr, va = next(GroupShuffleSplit(n_splits=1, test_size=.2, random_state=24).split(X, y, train.building_id))
assert set(train.iloc[tr].building_id).isdisjoint(train.iloc[va].building_id)
models = {'mean': make_model(DummyRegressor()), 'ridge': make_model(Ridge(alpha=10))}
comparison = []
for name, estimator in models.items():
    estimator.fit(X.iloc[tr], y.iloc[tr])
    pred = estimator.predict(X.iloc[va])
    comparison.append({'model': name, 'RMSE': float(mean_squared_error(y.iloc[va], pred) ** .5)})
comparison = pd.DataFrame(comparison)
print(comparison.to_string(index=False))

## Эксперименты


In [ ]:
validation_predictions = models['ridge'].predict(X.iloc[va])
residuals = y.iloc[va].to_numpy() - validation_predictions
print('Residual mean:', round(float(residuals.mean()), 5))
print('Residual quantiles:', np.quantile(residuals, [.05, .5, .95]).round(4))
print(pd.DataFrame({'type': train.iloc[va].building_type.to_numpy(), 'squared_error': residuals**2})
      .groupby('type').squared_error.mean().pow(.5).rename('RMSE').to_string())

## Выводы


In [ ]:
model = make_model(Ridge(alpha=10)).fit(X, y)
prediction = np.clip(model.predict(test.drop(columns=['row_id', 'building_id'])), 0, 50)
pd.DataFrame({'row_id': test.row_id, 'heat_kwh_m2': prediction}).to_csv('predictions.csv', index=False)
print('Saved predictions.csv:', len(prediction), 'rows')

## Сборка посылки


In [ ]:
participant = {
    'full_name': '',  # Фамилия, имя, отчество при наличии
    'university': '',  # ВШЭ или ИТМО
    'email': '',
    'codabench_username': '',
    'honor_code': False,  # True, если подтверждаете самостоятельность работы
}

if not all(participant.values()):
    print('Заполните анкету, сохраните ноутбук и выполните эту ячейку повторно.')
else:
    assert participant['university'] in {'ВШЭ', 'ИТМО'}, 'Укажите ВШЭ или ИТМО'
    assert len(participant['full_name'].split()) >= 2, 'Укажите фамилию и имя'
    assert '@' in participant['email'], 'Укажите адрес электронной почты'
    assert Path('solution.ipynb').is_file(), 'Сохраните ноутбук как solution.ipynb'
    Path('participant.json').write_text(
        json.dumps(participant, ensure_ascii=False, indent=2), encoding='utf-8'
    )
    with zipfile.ZipFile('submission.zip', 'w', zipfile.ZIP_DEFLATED) as archive:
        for name in ['predictions.csv', 'participant.json', 'solution.ipynb']:
            archive.write(name, arcname=name)
    print('Готово: загрузите submission.zip в Codabench.')